# CPI_通货膨胀：Hamilton Cycle 分段递推

本 notebook 使用 `output_宏观_pkl/M0000612.pkl` 的 CPI 当月同比，先用 Hamilton 回归法提取周期项 `hamilton_cycle`，再按固定长度分段。每一段会单独估计高/低通胀状态，并将本段参数递推到下一段。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()
PKL_DIR = ROOT / "output_宏观_pkl"
OUT = ROOT / "cpi_hamcycle_segment_recursive_output"
OUT.mkdir(exist_ok=True)

START_DATE = pd.Timestamp("2026-05-31")
CPI_CODE = "M0000612"
CPI_NAME = "中国:CPI:当月同比"

# 分段参数：60 个月约等于 5 年。可改为 36、48、72 做敏感性测试。
SEGMENT_MONTHS = 60
MIN_SEGMENT_OBS = 24

# Hamilton 参数：h 越小越敏感；p 是滞后阶数。
HAMILTON_H = 8
HAMILTON_P = 4

# 模型敏感度参数。
TRANSITION_STAY_INIT = 0.80
MIN_SWITCH_PROB = 0.08
LIKELIHOOD_SIGMA_SCALE = 0.85
STATE_PROB_THRESHOLD = 0.50
MAX_ITER = 500
TOL = 1e-8


## 读取 CPI 并生成 Hamilton 周期项

In [ ]:
def _first_numeric_series(obj) -> pd.Series:
    if isinstance(obj, pd.Series):
        return pd.to_numeric(obj, errors="coerce")
    if isinstance(obj, pd.DataFrame):
        numeric = obj.apply(pd.to_numeric, errors="coerce")
        cols = [col for col in numeric.columns if numeric[col].notna().any()]
        if not cols:
            raise ValueError("DataFrame 中没有可转成数值的列")
        return numeric[cols[0]]
    return pd.to_numeric(pd.Series(obj), errors="coerce")


def read_monthly_pkl(code: str, start_date: pd.Timestamp = START_DATE) -> pd.Series:
    path = PKL_DIR / f"{code}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    raw = _first_numeric_series(pd.read_pickle(path)).dropna()
    idx_text = pd.Series(raw.index.astype(str)).str.strip()
    parsed_idx = pd.to_datetime(idx_text, errors="coerce")

    if parsed_idx.notna().any():
        s = pd.Series(raw.to_numpy(), index=parsed_idx)
        s = s[s.index.notna()]
    else:
        dates = [start_date - pd.offsets.MonthEnd(i) for i in range(len(raw))]
        s = pd.Series(raw.to_numpy(), index=pd.DatetimeIndex(dates))

    month_end = pd.DatetimeIndex(s.index).to_period("M").to_timestamp("M")
    monthly = pd.Series(s.to_numpy(), index=month_end).sort_index().groupby(level=0).last()
    regular = monthly.reindex(pd.date_range(monthly.index.min(), monthly.index.max(), freq="ME"))
    return regular.interpolate(limit=2).dropna().rename(code)


def hamilton_filter(y: pd.Series, h: int = HAMILTON_H, p: int = HAMILTON_P) -> tuple[pd.Series, pd.Series]:
    x = y.astype(float).dropna()
    rows = []
    targets = []
    dates = []
    for t in range(p - 1, len(x) - h):
        rows.append([1.0] + [x.iloc[t - lag] for lag in range(p)])
        targets.append(x.iloc[t + h])
        dates.append(x.index[t + h])

    if len(rows) <= p:
        raise ValueError("Hamilton 回归样本不足")

    xmat = np.asarray(rows, dtype=float)
    yvec = np.asarray(targets, dtype=float)
    beta = np.linalg.lstsq(xmat, yvec, rcond=None)[0]
    trend = pd.Series(xmat @ beta, index=pd.DatetimeIndex(dates), name="hamilton_trend")
    cycle = (x.reindex(trend.index) - trend).rename("hamilton_cycle")
    return cycle, trend


raw_cpi = read_monthly_pkl(CPI_CODE)
ham_cycle, ham_trend = hamilton_filter(raw_cpi)

ham_data = pd.concat(
    {
        "raw_cpi": raw_cpi,
        "hamilton_trend": ham_trend,
        "hamilton_cycle": ham_cycle,
    },
    axis=1,
).rename_axis("date")

display(pd.DataFrame([{
    "code": CPI_CODE,
    "name": CPI_NAME,
    "raw_start": raw_cpi.index.min(),
    "raw_end": raw_cpi.index.max(),
    "ham_cycle_start": ham_cycle.index.min(),
    "ham_cycle_end": ham_cycle.index.max(),
    "ham_cycle_nobs": len(ham_cycle),
    "segment_months": SEGMENT_MONTHS,
}]))
display(ham_data.tail(12))


## 两状态模型函数

In [ ]:
def _normal_pdf(values: np.ndarray, mu: np.ndarray, sigma: float) -> np.ndarray:
    sigma = max(float(sigma), 1e-8)
    z = (values[:, None] - mu[None, :]) / sigma
    return np.exp(-0.5 * z * z) / (np.sqrt(2.0 * np.pi) * sigma)


def _constrain_transition(transition: np.ndarray, min_switch_prob: float) -> np.ndarray:
    transition = np.asarray(transition, dtype=float).copy()
    min_switch_prob = float(np.clip(min_switch_prob, 0.0, 0.49))
    max_stay_prob = 1.0 - min_switch_prob
    for i in range(transition.shape[0]):
        transition[i, i] = min(transition[i, i], max_stay_prob)
        off_diag = [j for j in range(transition.shape[1]) if j != i]
        remaining = 1.0 - transition[i, i]
        off_sum = transition[i, off_diag].sum()
        if off_sum <= 0:
            transition[i, off_diag] = remaining / len(off_diag)
        else:
            transition[i, off_diag] = transition[i, off_diag] / off_sum * remaining
    return transition / transition.sum(axis=1, keepdims=True)


def _params_to_arrays(initial_params: dict | None, values: np.ndarray) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    if initial_params is None:
        q25, q75 = np.nanpercentile(values, [25, 75])
        mu = np.asarray([q25, q75], dtype=float)
        sigma = float(np.nanstd(values, ddof=1)) or 1.0
        stay = float(np.clip(TRANSITION_STAY_INIT, 0.51, 0.99))
        transition = np.asarray([[stay, 1.0 - stay], [1.0 - stay, stay]], dtype=float)
        init_prob = np.asarray([0.50, 0.50], dtype=float)
        return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob

    mu = np.asarray([initial_params["mu_low"], initial_params["mu_high"]], dtype=float)
    sigma = float(initial_params.get("sigma", np.nanstd(values, ddof=1) or 1.0))
    transition = np.asarray(
        [
            [initial_params["p_stay_low"], initial_params["p_switch_low_to_high"]],
            [initial_params["p_switch_high_to_low"], initial_params["p_stay_high"]],
        ],
        dtype=float,
    )
    init_prob = np.asarray(initial_params.get("next_init_prob", [0.50, 0.50]), dtype=float)
    init_prob = init_prob / init_prob.sum()
    return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob


def _forward_filter(values: np.ndarray, mu: np.ndarray, sigma: float, transition: np.ndarray, init_prob: np.ndarray):
    emission = _normal_pdf(values, mu, sigma)
    prior = np.zeros((len(values), 2))
    filtered = np.zeros((len(values), 2))
    scale = np.zeros(len(values))

    prior[0] = init_prob
    filtered[0] = prior[0] * emission[0]
    scale[0] = filtered[0].sum()
    filtered[0] /= scale[0]

    for t in range(1, len(values)):
        prior[t] = filtered[t - 1] @ transition
        filtered[t] = prior[t] * emission[t]
        scale[t] = filtered[t].sum()
        filtered[t] /= scale[t]

    return prior, filtered, float(np.log(scale).sum())


def _state_frame(index: pd.DatetimeIndex, values: np.ndarray, prior: np.ndarray, filtered: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(index=index)
    out["model_value"] = values
    out["prior_prob_low_state"] = prior[:, 0]
    out["prior_prob_high_state"] = prior[:, 1]
    out["prob_low_state"] = filtered[:, 0]
    out["prob_high_state"] = filtered[:, 1]
    out["state"] = np.where(out["prob_high_state"] >= STATE_PROB_THRESHOLD, "高通胀状态", "低通胀状态")
    out["inflation_regime"] = np.where(out["prob_high_state"] >= STATE_PROB_THRESHOLD, "通胀偏高", "通胀偏低")
    return out


def fit_segment_hmm(y: pd.Series, initial_params: dict | None = None):
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    if len(values) < MIN_SEGMENT_OBS:
        raise ValueError(f"分段样本不足：{len(values)} < {MIN_SEGMENT_OBS}")

    mu, sigma, transition, init_prob = _params_to_arrays(initial_params, values)
    last_ll = -np.inf

    for iteration in range(1, MAX_ITER + 1):
        emission = _normal_pdf(values, mu, sigma)
        alpha = np.zeros((len(values), 2))
        scale = np.zeros(len(values))
        alpha[0] = init_prob * emission[0]
        scale[0] = alpha[0].sum()
        alpha[0] /= scale[0]

        for t in range(1, len(values)):
            alpha[t] = (alpha[t - 1] @ transition) * emission[t]
            scale[t] = alpha[t].sum()
            alpha[t] /= scale[t]

        beta = np.ones((len(values), 2))
        for t in range(len(values) - 2, -1, -1):
            beta[t] = transition @ (emission[t + 1] * beta[t + 1])
            beta[t] /= scale[t + 1]

        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)

        xi_sum = np.zeros((2, 2))
        for t in range(len(values) - 1):
            xi = alpha[t, :, None] * transition * emission[t + 1, None, :] * beta[t + 1, None, :]
            xi_sum += xi / xi.sum()

        init_prob = gamma[0]
        transition = _constrain_transition(xi_sum / xi_sum.sum(axis=1, keepdims=True), MIN_SWITCH_PROB)
        weights = gamma.sum(axis=0)
        mu = (gamma * values[:, None]).sum(axis=0) / weights
        sigma = np.sqrt(((gamma * (values[:, None] - mu[None, :]) ** 2).sum()) / len(values))

        ll = float(np.log(scale).sum())
        if abs(ll - last_ll) < TOL:
            break
        last_ll = ll

    order = np.argsort(mu)
    mu = mu[order]
    transition = transition[np.ix_(order, order)]
    init_prob = init_prob[order]
    init_prob = init_prob / init_prob.sum()

    sigma_for_filter = max(float(sigma) * LIKELIHOOD_SIGMA_SCALE, 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    states = _state_frame(x.index, values, prior, filtered)

    next_init_prob = filtered[-1] @ transition
    params = {
        "mu_low": float(mu[0]),
        "mu_high": float(mu[1]),
        "sigma": float(sigma),
        "sigma_for_filter": float(sigma_for_filter),
        "p_stay_low": float(transition[0, 0]),
        "p_switch_low_to_high": float(transition[0, 1]),
        "p_switch_high_to_low": float(transition[1, 0]),
        "p_stay_high": float(transition[1, 1]),
        "init_prob_low": float(init_prob[0]),
        "init_prob_high": float(init_prob[1]),
        "end_prob_low": float(filtered[-1, 0]),
        "end_prob_high": float(filtered[-1, 1]),
        "next_init_prob": next_init_prob / next_init_prob.sum(),
        "next_init_prob_low": float(next_init_prob[0]),
        "next_init_prob_high": float(next_init_prob[1]),
        "log_likelihood": filtered_ll,
        "iterations": iteration,
        "nobs": len(values),
    }
    return states, params


def apply_params_to_segment(y: pd.Series, params: dict) -> pd.DataFrame:
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    mu, sigma, transition, init_prob = _params_to_arrays(params, values)
    sigma_for_filter = max(float(sigma) * LIKELIHOOD_SIGMA_SCALE, 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    out = _state_frame(x.index, values, prior, filtered)
    out["log_likelihood_under_previous_params"] = filtered_ll
    return out


## 分段递推估计

In [ ]:
def make_segments(series: pd.Series, segment_months: int = SEGMENT_MONTHS) -> list[pd.Series]:
    x = series.dropna().sort_index()
    segments = []
    for start in range(0, len(x), segment_months):
        part = x.iloc[start : start + segment_months]
        if len(part) >= MIN_SEGMENT_OBS:
            segments.append(part)
    return segments


segments = make_segments(ham_cycle)

current_state_tables = []
next_eval_tables = []
segment_rows = []
parameter_rows = []

previous_params = None
for i, segment in enumerate(segments, start=1):
    segment_id = f"segment_{i:02d}"
    init_source = "sample_quantiles" if previous_params is None else f"segment_{i - 1:02d}_params"
    states, params = fit_segment_hmm(segment, initial_params=previous_params)

    states = states.rename_axis("date").reset_index()
    states.insert(0, "segment_id", segment_id)
    states.insert(1, "evaluation_type", "current_segment_refit")
    current_state_tables.append(states)

    segment_rows.append({
        "segment_id": segment_id,
        "init_source": init_source,
        "start": segment.index.min(),
        "end": segment.index.max(),
        "months": len(segment),
        "latest_date": states["date"].iloc[-1],
        "latest_prob_high_state": states["prob_high_state"].iloc[-1],
        "latest_state": states["state"].iloc[-1],
        "latest_inflation_regime": states["inflation_regime"].iloc[-1],
        "state_switches": int(states["state"].ne(states["state"].shift()).sum() - 1),
        "avg_abs_prob_high_change": states["prob_high_state"].diff().abs().mean(),
    })

    parameter_rows.append({
        "segment_id": segment_id,
        "init_source": init_source,
        "start": segment.index.min(),
        "end": segment.index.max(),
        **{k: v for k, v in params.items() if k != "next_init_prob"},
    })

    if i < len(segments):
        next_segment = segments[i]
        next_eval = apply_params_to_segment(next_segment, params)
        next_eval = next_eval.rename_axis("date").reset_index()
        next_eval.insert(0, "source_segment_id", segment_id)
        next_eval.insert(1, "target_segment_id", f"segment_{i + 1:02d}")
        next_eval.insert(2, "evaluation_type", "next_segment_using_previous_params")
        next_eval_tables.append(next_eval)

    previous_params = params

current_states = pd.concat(current_state_tables, ignore_index=True)
next_segment_evaluation = pd.concat(next_eval_tables, ignore_index=True) if next_eval_tables else pd.DataFrame()
segment_summary = pd.DataFrame(segment_rows)
parameter_trace = pd.DataFrame(parameter_rows)
latest_state = segment_summary.tail(1).copy()

display(segment_summary)
display(parameter_trace)
display(next_segment_evaluation.tail(20))


## 导出结果

In [ ]:
ham_data.to_csv(OUT / "cpi_hamilton_cycle_series.csv", encoding="utf-8-sig")
segment_summary.to_csv(OUT / "cpi_hamcycle_segment_summary.csv", index=False, encoding="utf-8-sig")
parameter_trace.to_csv(OUT / "cpi_hamcycle_parameter_trace.csv", index=False, encoding="utf-8-sig")
current_states.to_csv(OUT / "cpi_hamcycle_current_segment_states.csv", index=False, encoding="utf-8-sig")
next_segment_evaluation.to_csv(OUT / "cpi_hamcycle_next_segment_evaluation.csv", index=False, encoding="utf-8-sig")
latest_state.to_csv(OUT / "cpi_hamcycle_latest_state.csv", index=False, encoding="utf-8-sig")

excel_path = OUT / "cpi_hamcycle_segment_recursive.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    ham_data.reset_index().to_excel(writer, sheet_name="ham_cycle_series", index=False)
    segment_summary.to_excel(writer, sheet_name="segment_summary", index=False)
    parameter_trace.to_excel(writer, sheet_name="parameter_trace", index=False)
    current_states.to_excel(writer, sheet_name="current_segment_states", index=False)
    next_segment_evaluation.to_excel(writer, sheet_name="next_segment_eval", index=False)
    latest_state.to_excel(writer, sheet_name="latest_state", index=False)

print(f"已导出到：{OUT}")
print(f"Excel 文件：{excel_path}")
